# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

All dataset elements such as record sets, fields, and columns are referenced using their unique `@id` attributes for clarity and reproducibility.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and prepare to explore data using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset loaded: {metadata.name}")
print(f"Description: {metadata.description}")

# The metadata object is a structured instance; to view all attributes, use vars():
# print(json.dumps(vars(metadata), indent=2, default=str))  # Uncomment to see all metadata

## 2. Data Overview
Explore the available record sets (`@id`), their fields, and columns as defined in the Croissant schema.

**Tip:** All code in this notebook references objects using their unique `@id` for maximum clarity and reproducibility.

In [ ]:
# Show available record sets with their IDs
record_sets = dataset.record_sets
print("Record sets in the dataset:")
for rs in record_sets:
    print(f"  @id: {rs.id} | name: {rs.name}")
    fields = getattr(rs, 'fields', [])
    if fields:
        for field in fields:
            print(f"    - Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    else:
        print("    (No fields listed)")
    columns = getattr(rs, 'columns', [])
    if columns:
        for col in columns:
            print(f"    - Column @id: {col.id}, name: {col.name}")


## 3. Data Extraction
Extract data from record sets for analysis. Refer to the output above to get the specific `@id` of the record set(s) you want to analyze. Below, we demonstrate extracting all available record sets to pandas DataFrames, using their `@id`.

In [ ]:
# Create a dict for DataFrames by record set @id
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))  # Each is a dict with field @ids as keys
    if records:
        df = pd.DataFrame.from_records(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print(f"No records found for record set {rs_id}.")

# Preview the first record set if available
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nPreview of data for record set: {first_rs_id}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing: filtering, normalization, grouping. We work directly with field `@id`s as column names.

Below is a typical workflow; change `<numeric_field_id>` and `<group_field_id>` to match your data, referring to the Data Overview output above.

In [ ]:
# --- Customize these variables based on actual record set and field IDs --- #
# Use the code cell above to retrieve available record_set and field IDs.

# Example: Suppose first_rs_id = 'cr:recordSet/EVproteinSummary',
# numeric_field = 'cr:field/coveragePercent', group_field = 'cr:field/sampleType' (edit as needed)

if dataframes:
    # Select the first record set and try to infer a numeric field for demo
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    # Pick the first numeric field as default example
    numeric_field = numeric_fields[0] if numeric_fields else df.columns[0]

    print(f"Selected record_set_id: {record_set_id}")
    print(f"Selected numeric_field: {numeric_field}")

    threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fi' else None
    if threshold is not None:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} records")

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try to find a categorical/grouping field
        group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found for filtering/EDA.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Plot numeric data distributions or relationships. Uses pandas and matplotlib. Update field IDs as necessary.

In [ ]:
import matplotlib.pyplot as plt

if dataframes:
    df = dataframes[record_set_id]

    # Plot histogram for the selected numeric field
    if numeric_field in df.columns and df[numeric_field].dtype.kind in 'fi':
        plt.figure(figsize=(8, 4))
        df[numeric_field].hist(bins=30)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()

    # Boxplot grouped by categorical field if available
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 4))
        df.boxplot(column=numeric_field, by=group_field, grid=False, rot=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.show()


## 6. Conclusion
In this notebook, we loaded, inspected, filtered, and visualized the Mass Spectrometry Analysis of Extracellular Vesicles dataset using the `mlcroissant` library. All operations referenced dataset entities by their stable `@id` attributes for reproducibility and clarity. You can adapt this workflow to analyze other record sets, fields, and columns of interest in the dataset by updating the relevant `@id` values wherever needed.

- **Next steps:** Further domain-specific analyses, feature engineering, or ML modeling based on your research questions; consult the dataset's [FAIR2 metadata schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for deeper understanding of all available variables.

For more examples and advanced usage, see the [mlcroissant documentation](https://github.com/mlcommons/croissant).